In [ ]:
import random
import copy

class Node:
    def __init__(self, floor_state, position: tuple, parent, birth_action):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.cost = 0 #số ô sai của node = số ô sai của node + cost của parent
    
    def get_tuple_floor_state(self):
        return tuple(tuple(row) for row in self.floor_state)
    
class Queue:
    def __init__(self):
        self.queue = []
        
    def is_empty(self):
        return len(self.queue) == 0
    
    def enqueue(self, node):
        self.queue.append(node)
        
    def dequeue(self) -> Node:
        return self.queue.pop(0)
    
    def pop(self, index: int):
        return self.queue.pop(index)
        
    def pop_the_highest_priority(self) -> Node:
        #thêm 1 phương thức pop ưu tiên cao nhất (cost thấp nhất) trong queue
        max_priority_index = 0
        for i in range(len(self.queue)):
            if self.queue[i].cost < self.queue[max_priority_index].cost:
                max_priority_index = i
        return self.queue.pop(max_priority_index)
    
    def contain(self, other: Node):
        for i, node in enumerate(self.queue):
            if node.floor_state == other.floor_state and node.position == other.position:
                return i
        return -1


class Stack:
    def __init__(self):
        self.stack = []
    def is_empty(self):
        return len(self.stack) == 0
    def push(self, node: Node):
        self.stack.append(node)
    def pop(self) -> Node:
        return self.stack.pop()
    def contain(self, other: Node):
        for node in self.stack:
            if node.floor_state == other.floor_state and node.position == other.position:
                return True
        return False

GOAL_STATE = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
ROW, COL = 3, 3

def floor_and_vacuumpos_initialize():
    floor = []
    vacuum_pos = (random.randint(0, ROW - 1), random.randint(0, COL - 1))
    for i in range(ROW):
        floor.append([random.randint(0, 1) for _ in range(COL)])
    if floor[vacuum_pos[0]][vacuum_pos[1]] == 1:
        floor[vacuum_pos[0]][vacuum_pos[1]] = 0
    return floor, vacuum_pos

def posible_moves(vacuum_pos):
    moves = []
    if vacuum_pos[0] > 0: moves.append("UP")
    if vacuum_pos[0] < ROW - 1: moves.append("DOWN")
    if vacuum_pos[1] > 0: moves.append("LEFT")
    if vacuum_pos[1] < COL - 1: moves.append("RIGHT")
    return moves

def apply_move(floor, vacuum_pos, move):
    temp_floor = copy.deepcopy(floor)
    if move == "UP": temp_vac_pos = (vacuum_pos[0] - 1, vacuum_pos[1])
    elif move == "DOWN": temp_vac_pos = (vacuum_pos[0] + 1, vacuum_pos[1])
    elif move == "LEFT": temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] - 1)
    else: temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] + 1)
    
    if temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] == 1:
        temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] = 0
    return temp_floor, temp_vac_pos

def get_n0_of_dirty_tiles(floor: list) -> int:
    """
    Counting the number of dirty tiles
    """
    number_of_dirty_tile = 0
    for row in floor:
        for tile in row:
            if tile == 1:
                number_of_dirty_tile += 1
    return number_of_dirty_tile
                
def uniform_cost_first():
    floor, vacuum_pos = floor_and_vacuumpos_initialize()
    frontier = Queue()
    root = Node(floor_state=floor, position=vacuum_pos, parent=None, birth_action= None)
    root.cost= get_n0_of_dirty_tiles(floor)
    frontier.enqueue(root)
    visited_state = set()
    step = 0
    
    while True:
        if frontier.is_empty(): #dừng khi frontier trống (hết cách)
            print("Frontier is empty!!")
            break
        
        current_node = frontier.pop_the_highest_priority()
        step += 1
        if (current_node.get_tuple_floor_state(), current_node.position) not in visited_state: #check nếu state node có trong visited
            visited_state.add((current_node.get_tuple_floor_state(), current_node.position))
        else:
            continue
        
        if current_node.floor_state == GOAL_STATE: #dừng khi tìm thấy đáp án (GOAL)
            return current_node
        
        moves = posible_moves(current_node.position)
        for move in moves:
            # chạy thử từng bước có thể để sinh node con
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move)
            temp_node_cost = get_n0_of_dirty_tiles(temp_floor)
            temp_node = Node(temp_floor, temp_vacuum_pos, current_node, move) #sinh node con
            temp_node.cost = current_node.cost + temp_node_cost
            
            if frontier.contain(temp_node)== -1 and (temp_node.get_tuple_floor_state(), temp_vacuum_pos) not in visited_state:
                #kiểm tra nếu node con không có trong visited hoặc frontier
                frontier.enqueue(temp_node)
                
                #nếu có 1 node có cùng state nhưng cost tốt hơn thì thay thế node đó
            elif frontier.contain(temp_node) != -1:
                target_index = frontier.contain(temp_node)
                target_node = frontier.queue[target_index]
                if temp_node.cost < target_node.cost:
                    frontier.pop(target_index)
                    frontier.enqueue(temp_node)
            
                
def main():
    result = uniform_cost_first()
    if isinstance(result, Node):
        print(f"\nĐã tìm thấy đáp án (GOAL FOUND)!!!")
        
        # Trích xuất đường đi xuôi dòng từ Root đến Goal
        path = []
        current = result
        while current is not None:
            path.append(current)
            current = current.parent
        path.reverse()
        
        # In chi tiết từng bước di chuyển
        for step, node in enumerate(path):
            if step == 0:
                print(f"Bước xuất phát:")
            else:
                print(f"Bước {step}: Đi [{node.birth_action}]")
            for row in node.floor_state:
                print(row)
            print(f"Vị trí Robot: {node.position}")
            print("-" * 20)
    else:
        print(f"Không tìm thấy đáp án: {result}")

if __name__ == "__main__":
    main()


Đã tìm thấy đáp án (GOAL FOUND)!!!
Bước xuất phát:
[0, 0, 0]
[1, 1, 0]
[1, 1, 0]
Vị trí Robot: (0, 0)
--------------------
Bước 1: Đi [DOWN]
[0, 0, 0]
[0, 1, 0]
[1, 1, 0]
Vị trí Robot: (1, 0)
--------------------
Bước 2: Đi [DOWN]
[0, 0, 0]
[0, 1, 0]
[0, 1, 0]
Vị trí Robot: (2, 0)
--------------------
Bước 3: Đi [RIGHT]
[0, 0, 0]
[0, 1, 0]
[0, 0, 0]
Vị trí Robot: (2, 1)
--------------------
Bước 4: Đi [UP]
[0, 0, 0]
[0, 0, 0]
[0, 0, 0]
Vị trí Robot: (1, 1)
--------------------
